# Lecture02 発展: 用語集SFTと検索の比較

配布の用語集から作ったQ&AをLoRAで学習し、学習前後と検索を比較します。正解データは生成時には渡しません。

比較する4条件は **学習前・学習後・検索のみ・学習前モデル＋検索（RAG）** です。35問の小規模評価なので、普遍的な優劣は結論にしません。

Google ColabでT4 GPUを選び、上から順に実行してください。依存版・モデルのrevision・評価問題を固定しています。

## 0. 環境を準備する

配布ノートブックと同じ依存版を使います。学習済みの変数を残さないよう、再実行時はセッションを再起動してください。

In [ ]:
# @title 0. 固定版ライブラリの導入
import subprocess, sys
from pathlib import Path
PINS = {"transformers":"5.18.0", "peft":"0.21.1", "trl":"1.14.2", "accelerate":"1.15.0", "datasets":"5.1.0"}
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q",*[f"{k}=={v}" for k,v in PINS.items()]], check=True)
print("依存版:", PINS)


依存版: {'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0'}


## 1. 実験コードを読み込む

提出リポジトリの固定コミットを使用します。CLIからの実行時は同じファイルを事前に転送しています。

In [ ]:
# @title 1. リポジトリの準備
ROOT = Path("/content/aie_l02")
SOURCE_REVISION = "8f807bebbed72243eb460d0819b8fa8615e5164c"
if not (ROOT / "scripts/glossary_experiment.py").exists():
    subprocess.run(["git","clone","https://github.com/shikibuton10x/lecture-ai-engineering-2026-submission.git",str(ROOT)],check=True)
    subprocess.run(["git","-C",str(ROOT),"fetch","origin",SOURCE_REVISION],check=True)
    subprocess.run(["git","-C",str(ROOT),"checkout",SOURCE_REVISION],check=True)
sys.path.insert(0,str(ROOT / "scripts"))
print("実験ルート:", ROOT, "| コード版:", SOURCE_REVISION)


実験ルート: /content/aie_l02 | コード版: 8f807bebbed72243eb460d0819b8fa8615e5164c


## 2. 学習データと評価問題を固定する

配布スクリプトの116件に確認要求の手本16件を加えます。検証58件、評価35問とは質問文を分けます。同じ用語の定義は学習と評価で共有するため、これは未学習の用語そのものへの一般化を測る設計ではありません。

In [ ]:
# @title 2. データ作成と分割の確認
from prepare_glossary_experiment import prepare
manifest = prepare()


{
  "glossary_terms": 58,
  "raw_positive_examples": 116,
  "train_positive": 116,
  "train_abstention": 16,
  "train_total": 132,
  "valid": 58,
  "evaluation": 35,
  "evaluation_categories": {
    "description": 5,
    "direct": 10,
    "paraphrase": 10,
    "unknown_fictional": 4,
    "unknown_real": 6
  },
  "split": "同じ用語の定義は共有するが質問文はtrain/valid/testで非重複。評価問題とルーブリックは学習前に固定。",
  "sha256": {
    "glossary_sft.jsonl": "af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081",
    "train_messages.jsonl": "da8f986fbbd8a71fe615e115409fa3f364b348677c95ed13e0361dc070d3101c",
    "valid_messages.jsonl": "40630ccbfb7edda3a53f7dd2c70ca8d90ec475d3da603d1ed3f947f3a34af4e4",
    "evaluation_gold.jsonl": "dcbb5568b7996020cc273046c42771fca346b3f3992b2d71db5063283a6bd48f"
  }
}


## 3. 学習前のモデルと採点方法を用意する

モデルはQwen3.5-0.8Bのテキスト部分、32bit、量子化なしです。正解は採点時だけ参照します。概念語グループの一致は簡易指標なので、最後に回答原文も読みます。

In [ ]:
# @title 3. モデル読み込み・環境記録
from glossary_experiment import Experiment, CONFIG
experiment = Experiment()
print(CONFIG)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Dataset hashes and question split verified.
{
  "python": "3.13.16",
  "gpu": "Tesla T4",
  "gpu_total_gb": 15.637,
  "base_weight_gb": 3.01,
  "generation_eos_token_id": 248046,
  "generation_pad_token_id": 248044,
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "datasets": "5.1.0",
    "accelerate": "1.15.0"
  },
  "train_sequence_length": {
    "max": 180,
    "median": 132.5,
    "over_limit": 0
  }
}
{'base_model': 'Qwen/Qwen3.5-0.8B', 'revision': '2fc06364715b967f1860aea9cf38778875588b17', 'seed': 42, 'num_train_epochs': 10, 'lr': 0.0002, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'max_seq_len': 512, 'max_new_tokens': 192, 'batch_size': 4, 'gradient_accumulation_steps': 2, 'warmup_steps': 17, 'do_sample': False, 'method': 'LoRA, FP32, no quantization'}


## 4. 学習前を評価する

入力は質問と共通のシステム指示です。用語集本文は渡しません。

In [ ]:
# @title 4. 学習前の35問
before = experiment.evaluate("base_closedbook")


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


base_closedbook 1/35 direct:lora: 0 (7.9s)


base_closedbook 2/35 paraphrase:lora: 0 (5.2s)


... (30 outputs left out)

base_closedbook 33/35 unknown:07: 1 (4.6s)


base_closedbook 34/35 unknown:08: 0 (7.9s)


base_closedbook 35/35 unknown:09: 1 (6.5s)


{
  "method": "base_closedbook",
  "n": 35,
  "keyword_policy_rate": 0.37142857142857144,
  "known_keyword_rate": 0.36,
  "known_abstention_rate": 0.2,
  "unknown_abstention_rate": 0.4,
  "median_seconds": 6.48413,
  "total_seconds": 205.44255,
  "mean_output_tokens": 141.88571428571427,
  "by_category": {
    "direct": {
      "n": 10,
      "keyword_policy_rate": 0.3
    },
    "paraphrase": {
      "n": 10,
      "keyword_policy_rate": 0.5
    },
    "description": {
      "n": 5,
      "keyword_policy_rate": 0.2
    },
    "unknown_real": {
      "n": 6,
      "keyword_policy_rate": 0.3333333333333333
    },
    "unknown_fictional": {
      "n": 4,
      "keyword_policy_rate": 0.5
    }
  }
}


## 5. 用語集の検索だけで答える

質問の引用符内の用語、または用語集の名前・別名の最長一致から検索語を抽出し、配布のsearch_glossaryへ渡します。正解ラベルは使いません。用語名・別名の一致を中心とする検索のため、用語名を含まない質問で失敗する場合があります。これは実測する対象です。

In [ ]:
# @title 5. 配布の用語集検索
search = experiment.evaluate("search_only")


search_only 1/35 direct:lora: 1 (0.0s)


search_only 2/35 paraphrase:lora: 1 (0.0s)


search_only 3/35 description:lora: 0 (0.0s)


search_only 4/35 direct:qlora: 1 (0.0s)


search_only 5/35 paraphrase:qlora: 1 (0.0s)


search_only 6/35 direct:quantization: 1 (0.0s)


... (26 outputs left out)

search_only 33/35 unknown:07: 1 (0.0s)


search_only 34/35 unknown:08: 1 (0.0s)


search_only 35/35 unknown:09: 1 (0.0s)


{
  "method": "search_only",
  "n": 35,
  "keyword_policy_rate": 0.8571428571428571,
  "known_keyword_rate": 0.8,
  "known_abstention_rate": 0.16,
  "unknown_abstention_rate": 1.0,
  "median_seconds": 0.0007,
  "total_seconds": 0.03411,
  "mean_output_tokens": 0,
  "by_category": {
    "direct": {
      "n": 10,
      "keyword_policy_rate": 1.0
    },
    "paraphrase": {
      "n": 10,
      "keyword_policy_rate": 1.0
    },
    "description": {
      "n": 5,
      "keyword_policy_rate": 0.0
    },
    "unknown_real": {
      "n": 6,
      "keyword_policy_rate": 1.0
    },
    "unknown_fictional": {
      "n": 4,
      "keyword_policy_rate": 1.0
    }
  }
}


## 6. 学習前モデルに検索結果を渡す

同じ検索関数の結果を学習前モデルへ渡します。検索のみとの違いは、結果の解釈・文章生成をモデルが行う点です。

In [ ]:
# @title 6. RAGの35問
rag = experiment.evaluate("base_rag")


base_rag 1/35 direct:lora: 1 (5.9s)


base_rag 2/35 paraphrase:lora: 1 (3.3s)


base_rag 3/35 description:lora: 0 (5.0s)


base_rag 4/35 direct:qlora: 1 (3.5s)


base_rag 5/35 paraphrase:qlora: 1 (1.6s)


base_rag 6/35 direct:quantization: 1 (2.0s)


... (26 outputs left out)

base_rag 33/35 unknown:07: 1 (0.9s)


base_rag 34/35 unknown:08: 1 (0.4s)


base_rag 35/35 unknown:09: 1 (0.4s)


{
  "method": "base_rag",
  "n": 35,
  "keyword_policy_rate": 0.7714285714285715,
  "known_keyword_rate": 0.76,
  "known_abstention_rate": 0.16,
  "unknown_abstention_rate": 0.8,
  "median_seconds": 1.701,
  "total_seconds": 86.57127,
  "mean_output_tokens": 58.82857142857143,
  "by_category": {
    "direct": {
      "n": 10,
      "keyword_policy_rate": 0.9
    },
    "paraphrase": {
      "n": 10,
      "keyword_policy_rate": 0.9
    },
    "description": {
      "n": 5,
      "keyword_policy_rate": 0.2
    },
    "unknown_real": {
      "n": 6,
      "keyword_policy_rate": 0.6666666666666666
    },
    "unknown_fictional": {
      "n": 4,
      "keyword_policy_rate": 1.0
    }
  }
}


## 7. LoRAで教師あり学習する

132件を10周します。元モデルは固定し、r=16のadapterだけ更新します。学習時間、最後に表示されたloss、全学習の平均loss、終了時の検証lossを別々に記録します。

In [ ]:
# @title 7. LoRA学習と検証
training = experiment.train()


Tokenizing train dataset:   0%|          | 0/132 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/132 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/132 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/132 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

... (3 outputs left out)

trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.753991
20,1.398022
30,1.025105
40,0.728148
50,0.498928
60,0.246741
70,0.180762
80,0.110053
90,0.082308
100,0.069719


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.048758,0.426607,170,0.205921,175200.000000,0.946674


{
  "seconds": 592.669,
  "steps": 170,
  "epoch": 10.0,
  "trainable_parameters": 6389760,
  "total_parameters": 758782784,
  "trainable_percent": 0.8421066126877228,
  "train_mean_loss": 0.4385791047531016,
  "final_logged_loss": 0.04875811636447906,
  "validation": {
    "eval_loss": 0.4266073405742645,
    "eval_entropy": 0.2059208184480667,
    "eval_num_tokens": 175200.0,
    "eval_mean_token_accuracy": 0.9466742078463236
  },
  "peak_allocated_gpu_gb": 4.737959936
}


## 8. 同じ35問で学習後を評価する

学習前と質問・システム指示・生成設定・採点方法をそろえます。

In [ ]:
# @title 8. 学習後の35問
after = experiment.evaluate("sft_closedbook")


sft_closedbook 1/35 direct:lora: 1 (4.3s)


sft_closedbook 2/35 paraphrase:lora: 0 (0.7s)


sft_closedbook 3/35 description:lora: 0 (0.7s)


sft_closedbook 4/35 direct:qlora: 1 (4.0s)


sft_closedbook 5/35 paraphrase:qlora: 1 (5.3s)


sft_closedbook 6/35 direct:quantization: 1 (2.1s)


... (26 outputs left out)

sft_closedbook 33/35 unknown:07: 1 (0.7s)


sft_closedbook 34/35 unknown:08: 0 (4.1s)


sft_closedbook 35/35 unknown:09: 1 (0.7s)


{
  "method": "sft_closedbook",
  "n": 35,
  "keyword_policy_rate": 0.6857142857142857,
  "known_keyword_rate": 0.68,
  "known_abstention_rate": 0.12,
  "unknown_abstention_rate": 0.7,
  "median_seconds": 2.63508,
  "total_seconds": 94.11577,
  "mean_output_tokens": 48.8,
  "by_category": {
    "direct": {
      "n": 10,
      "keyword_policy_rate": 1.0
    },
    "paraphrase": {
      "n": 10,
      "keyword_policy_rate": 0.5
    },
    "description": {
      "n": 5,
      "keyword_policy_rate": 0.4
    },
    "unknown_real": {
      "n": 6,
      "keyword_policy_rate": 0.6666666666666666
    },
    "unknown_fictional": {
      "n": 4,
      "keyword_policy_rate": 0.75
    }
  }
}


## 9. 比較結果と原文を保存する

result.jsonに設定・環境・学習ログ・比較指標を保存します。comparison.csvには全質問の回答を並べます。用語集外への応答は、一般知識として正しくても「講座の用語集の範囲だけ答える」という運用方針への適合で採点します。

In [ ]:
# @title 9. 比較表と結果の保存
result = experiment.save_results()
import pandas as pd
display(pd.DataFrame(result["summaries"]).T[["n","keyword_policy_rate","known_keyword_rate","unknown_abstention_rate","median_seconds"]])
print("保存先:", ROOT / "lectures/02/advanced")


{
  "base_closedbook": {
    "method": "base_closedbook",
    "n": 35,
    "keyword_policy_rate": 0.37142857142857144,
    "known_keyword_rate": 0.36,
    "known_abstention_rate": 0.2,
    "unknown_abstention_rate": 0.4,
    "median_seconds": 6.48413,
    "total_seconds": 205.44255,
    "mean_output_tokens": 141.88571428571427,
    "by_category": {
      "direct": {
        "n": 10,
        "keyword_policy_rate": 0.3
      },
      "paraphrase": {
        "n": 10,
        "keyword_policy_rate": 0.5
      },
      "description": {
        "n": 5,
        "keyword_policy_rate": 0.2
      },
      "unknown_real": {
        "n": 6,
        "keyword_policy_rate": 0.3333333333333333
      },
      "unknown_fictional": {
        "n": 4,
        "keyword_policy_rate": 0.5
      }
    }
  },
  "search_only": {
    "method": "search_only",
    "n": 35,
    "keyword_policy_rate": 0.8571428571428571,
    "known_keyword_rate": 0.8,
    "known_abstention_rate": 0.16,
    "unknown_abstention_rate": 1

,n,keyword_policy_rate,known_keyword_rate,unknown_abstention_rate,median_seconds
base_closedbook,35,0.371429,0.36,0.4,6.48413
search_only,35,0.857143,0.8,1.0,0.0007
base_rag,35,0.771429,0.76,0.8,1.701
sft_closedbook,35,0.685714,0.68,0.7,2.63508


保存先: /content/aie_l02/lectures/02/advanced
